In [ ]:
!pip install torch torchvision matplotlib scikit-learn sympy==1.11.1 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 141.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.0/797.0 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 111.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 116.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 69.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.2/176.2 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
import csv
import time
import random
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchvision.transforms as T


In [ ]:
from pathlib import Path

In [ ]:
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, roc_auc_score, confusion_matrix
)

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

Mounted at /content/drive


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False


In [ ]:
DRIVE_BASE    = Path("/content/drive/MyDrive/AegisSafeRoad")
TENSORS_DIR   = DRIVE_BASE / "video_tensors"
MANIFEST_PATH = TENSORS_DIR / "manifest.csv"
TRAINING_DIR  = DRIVE_BASE / "training2"
TRAINING_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
BEST_MODEL_PATH = TRAINING_DIR / "best.pt"
ONNX_PATH       = TRAINING_DIR / "aegis_crash_detector.onnx"
PLOTS_DIR       = TRAINING_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH        = TRAINING_DIR / "training_log.csv"

In [ ]:
# HYPERPARAMS

N_FRAMES        = 16
IMG_C, IMG_H, IMG_W = 3, 224, 224
MOBILENET_DIM   = 1280       # MobileNetV2 penultimate layer output
ATTN_HIDDEN     = 512
ATTN_HIDDEN_2   = 128
DROPOUT         = 0.2

BATCH_SIZE      = 32
NUM_WORKERS     = 4
PIN_MEMORY      = True

# Stage 1: MobileNetV2 frozen — train MLP only
STAGE1_EPOCHS   = 4
LR_MLP_S1       = 3e-3

# Stage 2: last 4 layers MobileNetV2 unfrozen — fine-tune
STAGE2_EPOCHS   = 12
LR_MLP_S2       = 6e-4
LR_CNN_S2       = 5e-5

WEIGHT_DECAY    = 1e-4

# Early stopping
ES_PATIENCE     = 5          # epochs without improvement before stopping
ES_MIN_DELTA    = 1e-4       # minimum improvement threshold


In [ ]:
# ImageNet normalization applied in DataLoader
IMAGENET_MEAN   = [0.485, 0.456, 0.406]
IMAGENET_STD    = [0.229, 0.224, 0.225]


In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
print("=" * 60)
print("AegisSafeRoad — Training Pipeline")
print("=" * 60)
print(f"Device          : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Tensors dir     : {TENSORS_DIR}")
print(f"Training output : {TRAINING_DIR}")
print(f"Stage 1 epochs  : {STAGE1_EPOCHS}  (MLP only)")
print(f"Stage 2 epochs  : {STAGE2_EPOCHS}  (MLP + CNN fine-tune)")
print(f"Batch size      : {BATCH_SIZE}")
print(f"Early stopping  : patience={ES_PATIENCE}")
print("=" * 60)

AegisSafeRoad — Training Pipeline
Device          : cuda
GPU             : NVIDIA L4
VRAM            : 23.7 GB
Tensors dir     : /content/drive/MyDrive/AegisSafeRoad/video_tensors
Training output : /content/drive/MyDrive/AegisSafeRoad/training2
Stage 1 epochs  : 4  (MLP only)
Stage 2 epochs  : 12  (MLP + CNN fine-tune)
Batch size      : 32
Early stopping  : patience=5


In [ ]:
!rsync -ah --info=progress2 \
"/content/drive/MyDrive/AegisSafeRoad/video_tensors/" \
"/content/video_tensors/"

         29.65G 100%    4.06MB/s    1:56:10 (xfr#6157, to-chk=0/6161)


In [ ]:
!du -sh /content/video_tensors

28G	/content/video_tensors


In [ ]:
!find /content/video_tensors -name "*.npy" | wc -l

6156


In [ ]:
files = list(Path("/content/video_tensors/train").glob("*.npy"))[:1000]

t0 = time.time()

for f in files:
    x = np.load(f)

elapsed = time.time() - t0

print(f"1000 tensors loaded in {elapsed:.2f}s")
print(f"{1000/elapsed:.2f} tensors/sec")

1000 tensors loaded in 16.10s
62.10 tensors/sec


In [ ]:
#  DATASET

class CrashDataset(Dataset):
    """
    Reads pre-extracted (16, 3, 224, 224) float16 tensors from disk.
    Converts to float32 and applies ImageNet normalization on the fly.
    Each .npy file lives under video_tensors/{split}/.
    """

    NORMALIZE = T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)

    def __init__(self, manifest_path: Path, split: str, tensors_dir: Path):
        """
        Args:
            manifest_path : path to manifest.csv
            split         : "train" | "val" | "test"
            tensors_dir   : root directory containing train/ val/ test/
        """
        self.tensors_dir = tensors_dir
        self.records     = []

        with open(manifest_path, "r") as f:
            reader = csv.DictReader(f)
            for row in reader:
                if row["split"] == split:
                    self.records.append({
                        "file_path": row["file_path"],
                        "label"    : int(row["label"]),
                    })

        if len(self.records) == 0:
            raise RuntimeError(
                f"No records found for split='{split}' in {manifest_path}"
            )

        n_normal = sum(1 for r in self.records if r["label"] == 0)
        n_crash  = sum(1 for r in self.records if r["label"] == 1)

        print(f"[Dataset] split={split:5s}  "
              f"total={len(self.records)}  "
              f"normal={n_normal}  crash={n_crash}")

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, idx: int):
        record    = self.records[idx]
        npy_path  = self.tensors_dir / record["file_path"]
        label     = record["label"]

        # Load (16, 3, 224, 224) float16 -> float32
        tensor_f16 = np.load(str(npy_path))                          # (16,3,224,224)
        tensor_f32 = torch.from_numpy(tensor_f16.astype(np.float32)) # float32

        # Apply ImageNet normalization per frame
        # tensor_f32 shape: (16, 3, 224, 224)
        normalized = torch.stack([
            self.NORMALIZE(tensor_f32[i]) for i in range(N_FRAMES)
        ])  # (16, 3, 224, 224)

        return normalized, torch.tensor(label, dtype=torch.float32)


In [ ]:
#  TEMPRAL ATTENTION MLP
class TemporalAttention(nn.Module):
    """
    Temporal Attention pooling over N_FRAMES feature vectors.
    Learns which frames are most relevant for crash detection.

    Input  : (B, N_FRAMES, MOBILENET_DIM)  -> (B, 16, 1280)
    Output : (B, MOBILENET_DIM)            -> (B, 1280)  weighted sum
    """

    def __init__(self, input_dim: int = MOBILENET_DIM):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.Tanh(),
            nn.Linear(128, 1),          # score per frame
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, 16, 1280)
        scores  = self.attention(x)          # (B, 16, 1)
        weights = torch.softmax(scores, dim=1)  # (B, 16, 1)
        pooled  = (weights * x).sum(dim=1)   # (B, 1280)
        return pooled


In [ ]:
# CNN 2D MODEL + ATT-MLP
class CrashDetector(nn.Module):
    """
    Full model: MobileNetV2 feature extractor + Temporal Attention + MLP classifier.

    Forward pass:
        (B, 16, 3, 224, 224)
        -> per-frame MobileNetV2  -> (B, 16, 1280)
        -> TemporalAttention      -> (B, 1280)
        -> MLP classifier         -> (B, 1)
        -> sigmoid                -> probability [0, 1]
    """

    def __init__(self):
        super().__init__()

        # MobileNetV2 backbone — load ImageNet pretrained weights
        backbone = models.mobilenet_v2(
            weights=models.MobileNet_V2_Weights.IMAGENET1K_V1
        )

        # Remove classifier head — keep features + avgpool
        self.feature_extractor = nn.Sequential(
            backbone.features,  # MobileNetV2.features output: (B, 1280, 7, 7)
            nn.AdaptiveAvgPool2d((1, 1)),  # MobileNetV2.features[-1] is Conv2d 320->1280
            nn.Flatten(),               # (B, 1280)
        )

        # Freeze all layers initially (Stage 1)
        for param in self.feature_extractor.parameters():
            param.requires_grad = False

        # Temporal Attention
        self.temporal_attention = TemporalAttention(input_dim=MOBILENET_DIM)

        # MLP Classifier
        self.classifier = nn.Sequential(
            nn.Linear(MOBILENET_DIM, ATTN_HIDDEN),
            nn.ReLU(inplace=True),
            nn.Dropout(DROPOUT),
            nn.Linear(ATTN_HIDDEN, ATTN_HIDDEN_2),
            nn.ReLU(inplace=True),
            nn.Dropout(DROPOUT),
            nn.Linear(ATTN_HIDDEN_2, 1),
            nn.Sigmoid(),
        )

    def freeze_backbone(self):
        """Stage 1: freeze entire MobileNetV2."""
        for param in self.feature_extractor.parameters():
            param.requires_grad = False
        print("[Model] MobileNetV2 fully frozen")

    def unfreeze_last_n_layers(self, n: int = 4):
        """
        Stage 2: unfreeze last n layers of MobileNetV2 features.
        MobileNetV2.features has 19 sequential blocks (0-18).
        Unfreezing last 4: blocks 15, 16, 17, 18.
        """
        # First freeze everything
        for param in self.feature_extractor.parameters():
            param.requires_grad = False

        # Unfreeze last n blocks in backbone.features
        mobilenet_features = self.feature_extractor[0]  # nn.Sequential of blocks
        total_blocks = len(mobilenet_features)
        unfreeze_from = total_blocks - n

        for i, block in enumerate(mobilenet_features):
            if i >= unfreeze_from:
                for param in block.parameters():
                    param.requires_grad = True

        unfrozen_params = sum(
            p.numel() for p in self.feature_extractor.parameters()
            if p.requires_grad
        )
        print(f"[Model] MobileNetV2 last {n} layers unfrozen  "
              f"({unfrozen_params:,} trainable CNN params)")

    def get_param_groups(self, lr_mlp: float, lr_cnn: float) -> list:
        """
        Differential learning rates:
        CNN params (unfrozen) -> lr_cnn
        MLP + Attention params -> lr_mlp
        """
        cnn_params = [
            p for p in self.feature_extractor.parameters()
            if p.requires_grad
        ]
        mlp_params = (
            list(self.temporal_attention.parameters()) +
            list(self.classifier.parameters())
        )
        return [
            {"params": cnn_params, "lr": lr_cnn},
            {"params": mlp_params, "lr": lr_mlp},
        ]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, 16, 3, 224, 224)
        Returns:
            (B, 1) sigmoid probability
        """
        B, T, C, H, W = x.shape

        # Reshape to process all frames through MobileNetV2 in one batch
        x_flat   = x.view(B * T, C, H, W)               # (B*16, 3, 224, 224)
        features = self.feature_extractor(x_flat)         # (B*16, 1280)
        features = features.view(B, T, MOBILENET_DIM)     # (B, 16, 1280)

        # Temporal attention pooling
        pooled = self.temporal_attention(features)        # (B, 1280)

        # MLP classification
        out = self.classifier(pooled)                     # (B, 1)
        return out

    def count_parameters(self):

        total     = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return total, trainable


In [ ]:
# EARLY STOPPING:

class EarlyStopping:
    """
    Monitors validation loss and stops training when no improvement
    is observed for `patience` consecutive epochs.
    Saves the best model weights automatically.
    """

    def __init__(
        self,
        patience: int = ES_PATIENCE,
        min_delta: float = ES_MIN_DELTA,
        save_path: Path = BEST_MODEL_PATH,
    ):

        self.patience = patience
        self.min_delta = min_delta
        self.save_path = save_path

        self.best_loss = float("inf")
        self.counter = 0
        self.best_epoch = 0
        self.stop = False

    def step(self, val_loss: float, model: nn.Module, epoch: int):

        if val_loss < self.best_loss - self.min_delta:

            self.best_loss = val_loss
            self.counter = 0
            self.best_epoch = epoch

            torch.save(model.state_dict(), str(self.save_path))

            print(
                f"    [EarlyStopping] New best model saved  "
                f"val_loss={val_loss:.4f}  epoch={epoch}"
            )

        else:

            self.counter += 1

            print(
                f"    [EarlyStopping] No improvement  "
                f"counter={self.counter}/{self.patience}  "
                f"best={self.best_loss:.4f} @ epoch {self.best_epoch}"
            )

            if self.counter >= self.patience:

                self.stop = True

                print(
                    f"    [EarlyStopping] Triggered at epoch {epoch}. "
                    f"Best epoch was {self.best_epoch}."
                )

In [ ]:
def train_one_epoch(model: nn.Module,
                    loader: DataLoader,
                    optimizer: optim.Optimizer,
                    criterion: nn.Module,
                    device: torch.device) -> dict:
    model.train()
    total_loss = 0.0
    all_preds  = []
    all_labels = []

    for batch_idx, (tensors, labels) in enumerate(loader):
        tensors = tensors.to(device, non_blocking=True)   # (B,16,3,224,224)
        labels  = labels.to(device).unsqueeze(1)          # (B,1)

        optimizer.zero_grad()
        outputs = model(tensors)                           # (B,1)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * tensors.size(0)
        preds = (outputs.detach().cpu() >= 0.5).float().squeeze(1)
        all_preds.extend(preds.numpy())
        all_labels.extend(labels.cpu().squeeze(1).numpy())

    n         = len(loader.dataset)
    avg_loss  = total_loss / n
    accuracy  = accuracy_score(all_labels, all_preds)
    f1        = f1_score(all_labels, all_preds, zero_division=0)
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall    = recall_score(all_labels, all_preds, zero_division=0)

    return {"loss": avg_loss, "accuracy": accuracy, "f1": f1, "precision": precision, "recall": recall}

In [ ]:
# EVALUATE

@torch.no_grad()
def evaluate(model: nn.Module,
             loader: DataLoader,
             criterion: nn.Module,
             device: torch.device) -> dict:
    model.eval()
    total_loss = 0.0
    all_preds  = []
    all_probs  = []
    all_labels = []

    for tensors, labels in loader:
        tensors = tensors.to(device, non_blocking=True)
        labels  = labels.to(device).unsqueeze(1)

        outputs    = model(tensors)
        loss       = criterion(outputs, labels)
        total_loss += loss.item() * tensors.size(0)

        probs = outputs.cpu().squeeze(1)
        preds = (probs >= 0.5).float()
        all_probs.extend(probs.numpy())
        all_preds.extend(preds.numpy())
        all_labels.extend(labels.cpu().squeeze(1).numpy())

    n        = len(loader.dataset)
    avg_loss = total_loss / n
    accuracy = accuracy_score(all_labels, all_preds)
    f1       = f1_score(all_labels, all_preds, zero_division=0)
    precision= precision_score(all_labels, all_preds, zero_division=0)
    recall   = recall_score(all_labels, all_preds, zero_division=0)
    auc      = roc_auc_score(all_labels, all_probs)
    cm       = confusion_matrix(all_labels, all_preds)

    return {
        "loss"     : avg_loss,
        "accuracy" : accuracy,
        "f1"       : f1,
        "precision": precision,
        "recall"   : recall,
        "auc"      : auc,
        "cm"       : cm,
    }


In [ ]:
# PLOTTING

def plot_training_curves(history: dict, plots_dir: Path, stage: str):
    """
    Saves training curves for a given stage:
    - Loss (train + val)
    - Accuracy (train + val)
    - F1-Score (train + val)
    - Precision (train + val)
    - Recall (train + val)
    - Learning Rate
    """
    epochs = list(range(1, len(history["train_loss"]) + 1))
    fig, axes = plt.subplots(3, 2, figsize=(16, 15)) # Changed to 3 rows to accommodate more plots
    fig.suptitle(f"AegisSafeRoad — Training Curves ({stage})", fontsize=14)

    # Loss
    ax = axes[0, 0]
    ax.plot(epochs, history["train_loss"], label="Train Loss", marker="o")
    ax.plot(epochs, history["val_loss"],   label="Val Loss",   marker="s")
    ax.set_title("Loss")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("BCE Loss")
    ax.legend()
    ax.grid(True)

    # Accuracy
    ax = axes[0, 1]
    ax.plot(epochs, history["train_acc"], label="Train Accuracy", marker="o")
    ax.plot(epochs, history["val_acc"],   label="Val Accuracy",   marker="s")
    ax.set_title("Accuracy")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy")
    ax.set_ylim([0, 1])
    ax.legend()
    ax.grid(True)

    # F1 Score
    ax = axes[1, 0]
    ax.plot(epochs, history["train_f1"], label="Train F1", marker="o")
    ax.plot(epochs, history["val_f1"],   label="Val F1",   marker="s")
    ax.set_title("F1-Score")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("F1")
    ax.set_ylim([0, 1])
    ax.legend()
    ax.grid(True)

    # Precision
    ax = axes[1, 1]
    ax.plot(epochs, history["train_precision"], label="Train Precision", marker="o")
    ax.plot(epochs, history["val_precision"],   label="Val Precision",   marker="s")
    ax.set_title("Precision")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Precision")
    ax.set_ylim([0, 1])
    ax.legend()
    ax.grid(True)

    # Recall
    ax = axes[2, 0]
    ax.plot(epochs, history["train_recall"], label="Train Recall", marker="o")
    ax.plot(epochs, history["val_recall"],   label="Val Recall",   marker="s")
    ax.set_title("Recall")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Recall")
    ax.set_ylim([0, 1])
    ax.legend()
    ax.grid(True)

    # Learning Rate
    ax = axes[2, 1]
    ax.plot(epochs, history["lr"], label="Learning Rate", marker="o", color="purple")
    ax.set_title("Learning Rate")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("LR")
    ax.set_yscale("log")
    ax.legend()
    ax.grid(True)

    plt.tight_layout()
    path = plots_dir / f"curves_{stage}.png"
    plt.savefig(str(path), dpi=120, bbox_inches="tight")
    plt.close()
    print(f"[Plot] Saved: {path}")



In [ ]:

def plot_confusion_matrix(cm: np.ndarray, plots_dir: Path, title: str):
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
    plt.colorbar(im, ax=ax)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["Normal", "Crash"])
    ax.set_yticklabels(["Normal", "Crash"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]),
                    ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black",
                    fontsize=14)
    plt.tight_layout()
    path = plots_dir / f"confusion_matrix_{title.replace(' ', '_')}.png"
    plt.savefig(str(path), dpi=120, bbox_inches="tight")
    plt.close()
    print(f"[Plot] Saved: {path}")



In [ ]:

def plot_combined_curves(history_s1: dict, history_s2: dict, plots_dir: Path):
    """Combined loss and accuracy across both stages."""
    train_loss = history_s1["train_loss"] + history_s2["train_loss"]
    val_loss   = history_s1["val_loss"]   + history_s2["val_loss"]
    train_acc  = history_s1["train_acc"]  + history_s2["train_acc"]
    val_acc    = history_s1["val_acc"]    + history_s2["val_acc"]
    train_f1   = history_s1["train_f1"]   + history_s2["train_f1"] # Added train_f1
    val_f1     = history_s1["val_f1"]     + history_s2["val_f1"]
    train_precision = history_s1["train_precision"] + history_s2["train_precision"]
    val_precision   = history_s1["val_precision"]   + history_s2["val_precision"]
    train_recall    = history_s1["train_recall"]    + history_s2["train_recall"]
    val_recall      = history_s1["val_recall"]      + history_s2["val_recall"]

    total_ep   = len(train_loss)
    epochs     = list(range(1, total_ep + 1))
    s1_end     = STAGE1_EPOCHS

    fig, axes = plt.subplots(2, 3, figsize=(18, 10)) # Changed to 2 rows, 3 columns
    fig.suptitle("AegisSafeRoad — Full Training (Stage 1 + Stage 2)", fontsize=13)

    # Loss
    ax = axes[0, 0]
    ax.plot(epochs, train_loss, label="Train", marker="o")
    ax.plot(epochs, val_loss,   label="Val",   marker="s")
    ax.axvline(x=s1_end + 0.5, color="red", linestyle="--",
               linewidth=1.5, label="Stage 1 | Stage 2")
    ax.set_title("Loss")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("BCE Loss")
    ax.legend()
    ax.grid(True)

    # Accuracy
    ax = axes[0, 1]
    ax.plot(epochs, train_acc, label="Train", marker="o")
    ax.plot(epochs, val_acc,   label="Val",   marker="s")
    ax.axvline(x=s1_end + 0.5, color="red", linestyle="--",
               linewidth=1.5, label="Stage 1 | Stage 2")
    ax.set_title("Accuracy")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy")
    ax.set_ylim([0, 1])
    ax.legend()
    ax.grid(True)

    # F1-Score
    ax = axes[0, 2]
    ax.plot(epochs, train_f1, label="Train F1", marker="o") # Added train F1 for combined plots
    ax.plot(epochs, val_f1,   label="Val F1",   marker="s")
    ax.axvline(x=s1_end + 0.5, color="red", linestyle="--",
               linewidth=1.5, label="Stage 1 | Stage 2")
    ax.set_title("F1-Score")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("F1-Score")
    ax.set_ylim([0, 1])
    ax.legend()
    ax.grid(True)

    # Precision
    ax = axes[1, 0]
    ax.plot(epochs, train_precision, label="Train Precision", marker="o")
    ax.plot(epochs, val_precision,   label="Val Precision",   marker="s")
    ax.axvline(x=s1_end + 0.5, color="red", linestyle="--",
               linewidth=1.5, label="Stage 1 | Stage 2")
    ax.set_title("Precision")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Precision")
    ax.set_ylim([0, 1])
    ax.legend()
    ax.grid(True)

    # Recall
    ax = axes[1, 1]
    ax.plot(epochs, train_recall, label="Train Recall", marker="o")
    ax.plot(epochs, val_recall,   label="Val Recall",   marker="s")
    ax.axvline(x=s1_end + 0.5, color="red", linestyle="--",
               linewidth=1.5, label="Stage 1 | Stage 2")
    ax.set_title("Recall")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Recall")
    ax.set_ylim([0, 1])
    ax.legend()
    ax.grid(True)

    plt.tight_layout()
    path = plots_dir / "curves_combined.png"
    plt.savefig(str(path), dpi=120, bbox_inches="tight")
    plt.close()
    print(f"[Plot] Saved: {path}")

In [ ]:
# VSV LOGS

def init_log(log_path: Path):
    with open(log_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "stage", "epoch", "train_loss", "train_acc", "train_f1",
            "val_loss", "val_acc", "val_f1", "val_precision",
            "val_recall", "val_auc", "lr", "epoch_time_s"
        ])



In [ ]:
# LOG KEEPER

def append_log(log_path: Path, row: dict):
    with open(log_path, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            row["stage"], row["epoch"],
            f"{row['train_loss']:.6f}", f"{row['train_acc']:.4f}",
            f"{row['train_f1']:.4f}",  f"{row['val_loss']:.6f}",
            f"{row['val_acc']:.4f}",   f"{row['val_f1']:.4f}",
            f"{row['val_precision']:.4f}", f"{row['val_recall']:.4f}",
            f"{row['val_auc']:.4f}",   f"{row['lr']:.2e}",
            f"{row['epoch_time_s']:.1f}",
        ])



In [ ]:
#EXPORT TO ONNX

def export_onnx(model: nn.Module, save_path: Path, device: torch.device):
    """
    Export full model to ONNX.
    Input  : (1, 16, 3, 224, 224)  float32
    Output : (1, 1)                float32  sigmoid probability
    """
    model.eval()
    dummy = torch.randn(1, N_FRAMES, IMG_C, IMG_H, IMG_W).to(device)

    torch.onnx.export(
        model,
        dummy,
        str(save_path),
        export_params   = True,
        opset_version   = 17,
        do_constant_folding = True,
        input_names     = ["frames"],
        output_names    = ["crash_probability"],
        dynamic_axes    = {
            "frames"           : {0: "batch_size"},
            "crash_probability": {0: "batch_size"},
        },
    )
    size_mb = save_path.stat().st_size / 1e6
    print(f"[ONNX] Exported to: {save_path}  ({size_mb:.1f} MB)")



In [ ]:
# TRAINING PIPELINE

def run_stage(stage_name: str,
              model: nn.Module,
              optimizer: optim.Optimizer,
              scheduler,
              n_epochs: int,
              train_loader: DataLoader,
              val_loader: DataLoader,
              criterion: nn.Module,
              early_stopping: EarlyStopping,
              log_path: Path,
              global_epoch_offset: int = 0) -> dict:
    """
    Runs one training stage.
    Returns history dict with per-epoch metrics.
    """
    history = {
        "train_loss": [], "train_acc": [], "train_f1": [],
        "train_precision": [], "train_recall": [], # Added precision and recall
        "val_loss"  : [], "val_acc" : [], "val_f1"  : [],
        "val_precision": [], "val_recall": [], # Added precision and recall
        "lr"        : [],
    }

    print(f"\n{'='*60}")
    print(f"  {stage_name}")
    print(f"{'='*60}")

    for epoch in range(1, n_epochs + 1):
        global_epoch = global_epoch_offset + epoch
        t0           = time.time()

        # Current LR (first param group — MLP or CNN depending on stage)
        current_lr = optimizer.param_groups[-1]["lr"]

        # Train
        train_metrics = train_one_epoch(
            model, train_loader, optimizer, criterion, DEVICE
        )

        # Validate
        val_metrics = evaluate(model, val_loader, criterion, DEVICE)

        # Scheduler step
        scheduler.step()

        # Elapsed time
        elapsed = time.time() - t0

        # Log to console
        print(
            f"  [{stage_name}] Epoch {epoch:02d}/{n_epochs}  "
            f"| train_loss={train_metrics['loss']:.4f}  "
            f"train_acc={train_metrics['accuracy']:.4f}  "
            f"train_f1={train_metrics['f1']:.4f}  "
            f"train_prec={train_metrics['precision']:.4f}  " \
            f"train_rec={train_metrics['recall']:.4f}  " \
            f"| val_loss={val_metrics['loss']:.4f}  "
            f"val_acc={val_metrics['accuracy']:.4f}  "
            f"val_f1={val_metrics['f1']:.4f}  "
            f"val_prec={val_metrics['precision']:.4f}  " \
            f"val_rec={val_metrics['recall']:.4f}  " \
            f"val_auc={val_metrics['auc']:.4f}  "
            f"| lr={current_lr:.2e}  "
            f"| {elapsed:.0f}s"
        )

        # Append to history
        history["train_loss"].append(train_metrics["loss"])
        history["train_acc"] .append(train_metrics["accuracy"])
        history["train_f1"]  .append(train_metrics["f1"])
        history["train_precision"].append(train_metrics["precision"]) # Added to history
        history["train_recall"] .append(train_metrics["recall"])    # Added to history
        history["val_loss"]  .append(val_metrics["loss"])
        history["val_acc"]   .append(val_metrics["accuracy"])
        history["val_f1"]    .append(val_metrics["f1"])
        history["val_precision"].append(val_metrics["precision"])   # Added to history
        history["val_recall"] .append(val_metrics["recall"])      # Added to history
        history["lr"]        .append(current_lr)

        # CSV log
        append_log(log_path, {
            "stage"       : stage_name,
            "epoch"       : global_epoch,
            "train_loss"  : train_metrics["loss"],
            "train_acc"   : train_metrics["accuracy"],
            "train_f1"    : train_metrics["f1"],
            "train_precision": train_metrics["precision"], # Added to log
            "train_recall"  : train_metrics["recall"],
            "val_loss"    : val_metrics["loss"],
            "val_acc"     : val_metrics["accuracy"],
            "val_f1"      : val_metrics["f1"],
            "val_precision": val_metrics["precision"],
            "val_recall"  : val_metrics["recall"],
            "val_auc"     : val_metrics["auc"],
            "lr"          : current_lr,
            "epoch_time_s": elapsed,
        })

        # Early stopping check
        early_stopping.step(val_metrics["loss"], model, global_epoch)
        if early_stopping.stop:
            print(f"  [{stage_name}] Early stopping triggered at epoch {epoch}.")
            break

    return history

In [ ]:
import shutil
import subprocess

In [ ]:
COLAB_TENSORS = Path("/content/video_tensors")

In [ ]:
TENSORS_DIR   = COLAB_TENSORS
MANIFEST_PATH = TENSORS_DIR / "manifest.csv"
print(f"[Cache] Training will read from: {TENSORS_DIR}")

[Cache] Training will read from: /content/video_tensors


In [ ]:
print(TENSORS_DIR)

/content/video_tensors


In [ ]:
print(MANIFEST_PATH)

/content/video_tensors/manifest.csv


In [ ]:

result = subprocess.run(
    ["du", "-sh", "/content/video_tensors/"],
    capture_output=True, text=True
)
print(result.stdout)

# Count files copied so far
npy_count = len(list(Path("/content/video_tensors").rglob("*.npy")))
print(f"Files copied so far: {npy_count} / 6156")

28G	/content/video_tensors/

Files copied so far: 6156 / 6156


In [ ]:
# MAIN TRAINING LOOP

def main():

    # i. Build datasets and dataloaders

    print("[Step 1] Building datasets...")

    train_dataset = CrashDataset(MANIFEST_PATH, "train", TENSORS_DIR)
    val_dataset   = CrashDataset(MANIFEST_PATH, "val",   TENSORS_DIR)
    test_dataset  = CrashDataset(MANIFEST_PATH, "test",  TENSORS_DIR)

    train_loader = DataLoader(
        train_dataset,
        batch_size  = BATCH_SIZE,
        shuffle     = True,
        num_workers = NUM_WORKERS,
        pin_memory  = PIN_MEMORY,
        drop_last   = True,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size  = BATCH_SIZE,
        shuffle     = False,
        num_workers = NUM_WORKERS,
        pin_memory  = PIN_MEMORY,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size  = BATCH_SIZE,
        shuffle     = False,
        num_workers = NUM_WORKERS,
        pin_memory  = PIN_MEMORY,
    )

    print(f"[Step 1] Train batches : {len(train_loader)}")
    print(f"[Step 1] Val batches   : {len(val_loader)}")
    print(f"[Step 1] Test batches  : {len(test_loader)}")

    # ii. Build model

    print("\n[Step 2] Building model...")
    model = CrashDetector().to(DEVICE)
    model.freeze_backbone()

    total_params, trainable_params = model.count_parameters()
    print(f"[Step 2] Total params     : {total_params:,}")
    print(f"[Step 2] Trainable params : {trainable_params:,}")

    # Loss — BCELoss (model outputs sigmoid)
    criterion = nn.BCELoss()

    # Init CSV log
    init_log(LOG_PATH)

    # Early stopping — shared across both stages
    early_stopping = EarlyStopping(
        patience  = ES_PATIENCE,
        min_delta = ES_MIN_DELTA,
        save_path = BEST_MODEL_PATH,
    )


    # iii. Stage 1 — Train MLP Attention only (MobileNetV2 frozen)

    print("\n[Step 3] Stage 1 — MLP Attention only (MobileNetV2 frozen)")

    optimizer_s1 = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr           = LR_MLP_S1,
        weight_decay = WEIGHT_DECAY,
    )
    scheduler_s1 = optim.lr_scheduler.CosineAnnealingLR(
        optimizer_s1,
        T_max  = STAGE1_EPOCHS,
        eta_min= LR_MLP_S1 * 0.1,
    )

    history_s1 = run_stage(
        stage_name          = "Stage1_MLP",
        model               = model,
        optimizer           = optimizer_s1,
        scheduler           = scheduler_s1,
        n_epochs            = STAGE1_EPOCHS,
        train_loader        = train_loader,
        val_loader          = val_loader,
        criterion           = criterion,
        early_stopping      = early_stopping,
        log_path            = LOG_PATH,
        global_epoch_offset = 0,
    )

    plot_training_curves(history_s1, PLOTS_DIR, "Stage1_MLP")

    if early_stopping.stop:
        print("[Warning] Early stopping triggered in Stage 1. Skipping Stage 2.")
    else:

        # iv. Stage 2 — Fine-tune last 4 MobileNetV2 layers + MLP

        print("\n[Step 4] Stage 2 — Unfreezing last 4 MobileNetV2 layers")
        model.unfreeze_last_n_layers(n=4)

        total_params, trainable_params = model.count_parameters()
        print(f"[Step 4] Trainable params now : {trainable_params:,}")

        # Differential LR: CNN layers get lr_cnn, MLP gets lr_mlp
        optimizer_s2 = optim.Adam(
            model.get_param_groups(lr_mlp=LR_MLP_S2, lr_cnn=LR_CNN_S2),
            weight_decay = WEIGHT_DECAY,
        )
        scheduler_s2 = optim.lr_scheduler.CosineAnnealingLR(
            optimizer_s2,
            T_max  = STAGE2_EPOCHS,
            eta_min= LR_CNN_S2 * 0.1,
        )

        # Reset early stopping counter for Stage 2
        # (keep best_loss from Stage 1 so Stage 2 must beat it)
        early_stopping.counter = 0
        early_stopping.stop    = False

        history_s2 = run_stage(
            stage_name          = "Stage2_FineTune",
            model               = model,
            optimizer           = optimizer_s2,
            scheduler           = scheduler_s2,
            n_epochs            = STAGE2_EPOCHS,
            train_loader        = train_loader,
            val_loader          = val_loader,
            criterion           = criterion,
            early_stopping      = early_stopping,
            log_path            = LOG_PATH,
            global_epoch_offset = STAGE1_EPOCHS,
        )

        plot_training_curves(history_s2, PLOTS_DIR, "Stage2_FineTune")
        plot_combined_curves(history_s1, history_s2, PLOTS_DIR)


    # v. Load best model and evaluate on test set

    print("\n[Step 5] Loading best model for test evaluation...")
    model.load_state_dict(torch.load(str(BEST_MODEL_PATH), map_location=DEVICE))
    model.eval()

    test_metrics = evaluate(model, test_loader, criterion, DEVICE)

    print("\n" + "=" * 60)
    print("  TEST SET RESULTS")
    print("=" * 60)
    print(f"  Accuracy  : {test_metrics['accuracy']:.4f}")
    print(f"  F1-Score  : {test_metrics['f1']:.4f}")
    print(f"  Precision : {test_metrics['precision']:.4f}")
    print(f"  Recall    : {test_metrics['recall']:.4f}")
    print(f"  AUC-ROC   : {test_metrics['auc']:.4f}")
    print(f"  Confusion Matrix:")
    print(f"    {test_metrics['cm']}")
    print("=" * 60)

    plot_confusion_matrix(
        test_metrics["cm"], PLOTS_DIR, "Test Set Final"
    )

    # Save test results to txt
    test_results_path = TRAINING_DIR / "test_results.txt"
    with open(test_results_path, "w") as f:
        f.write("AegisSafeRoad — Test Set Results\n")
        f.write("=" * 40 + "\n") # Fixed the SyntaxError here
        f.write(f"Accuracy  : {test_metrics['accuracy']:.4f}\n")
        f.write(f"F1-Score  : {test_metrics['f1']:.4f}\n")
        f.write(f"Precision : {test_metrics['precision']:.4f}\n")
        f.write(f"Recall    : {test_metrics['recall']:.4f}\n")
        f.write(f"AUC-ROC   : {test_metrics['auc']:.4f}\n")
        f.write(f"Best epoch: {early_stopping.best_epoch}\n")
        f.write(f"Best val loss: {early_stopping.best_loss:.4f}\n")
    print(f"[Step 5] Test results saved: {test_results_path}")

    # vii. Final summary

    print(f"""
{'='*60}
  TRAINING COMPLETE
{'='*60}
  Best model  : {BEST_MODEL_PATH}
  ONNX export : {ONNX_PATH}
  Training log: {LOG_PATH}
  Plots       : {PLOTS_DIR}
  Test results: {test_results_path}

  Test Accuracy : {test_metrics['accuracy']:.4f}
  Test F1-Score : {test_metrics['f1']:.4f}
  Test AUC-ROC  : {test_metrics['auc']:.4f}
  Best epoch    : {early_stopping.best_epoch}
{'='*60}
""")

    return model

In [ ]:
# RUN

model = main()

[Step 1] Building datasets...
[Dataset] split=train  total=5039  normal=2519  crash=2520
[Dataset] split=val    total=1000  normal=500  crash=500
[Dataset] split=test   total=117  normal=60  crash=57
[Step 1] Train batches : 157
[Step 1] Val batches   : 32
[Step 1] Test batches  : 4

[Step 2] Building model...
[Model] MobileNetV2 fully frozen
[Step 2] Total params     : 3,109,634
[Step 2] Trainable params : 885,762

[Step 3] Stage 1 — MLP Attention only (MobileNetV2 frozen)

  Stage1_MLP
  [Stage1_MLP] Epoch 01/4  | train_loss=0.2759  train_acc=0.8836  train_f1=0.8836  train_prec=0.8827  train_rec=0.8845  | val_loss=0.0976  val_acc=0.9680  val_f1=0.9677  val_prec=0.9776  val_rec=0.9580  val_auc=0.9937  | lr=3.00e-03  | 83s
    [EarlyStopping] New best model saved  val_loss=0.0976  epoch=1
  [Stage1_MLP] Epoch 02/4  | train_loss=0.1261  train_acc=0.9572  train_f1=0.9572  train_prec=0.9570  train_rec=0.9574  | val_loss=0.0960  val_acc=0.9690  val_f1=0.9696  val_prec=0.9501  val_rec=0.990

/tmp/ipykernel_2174/2870819895.py:144: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(str(BEST_MODEL_PATH), map_location=DEVICE))



  TEST SET RESULTS
  Accuracy  : 0.9829
  F1-Score  : 0.9828
  Precision : 0.9661
  Recall    : 1.0000
  AUC-ROC   : 0.9950
  Confusion Matrix:
    [[58  2]
 [ 0 57]]
[Plot] Saved: /content/drive/MyDrive/AegisSafeRoad/training2/plots/confusion_matrix_Test_Set_Final.png
[Step 5] Test results saved: /content/drive/MyDrive/AegisSafeRoad/training2/test_results.txt

[Step 6] Exporting model to ONNX...


OnnxExporterError: Module onnx is not installed!

In [ ]:
best_model_path = "content/dirve/MyDrive/AegisSafeRoad/training/best.pt"

In [ ]:
!pip install onnx -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 111.8 MB/s eta 0:00:00


In [ ]:
# vi. Export to ONNX
# Load the best model and export it to ONNX without rerunning the main training loop.
print("\n[Step 6] Exporting model to ONNX...")

# Instantiate the model
model = CrashDetector().to(DEVICE)

# Load the best weights
model.load_state_dict(torch.load(str(BEST_MODEL_PATH), map_location=DEVICE))
model.eval()

# Export to ONNX
export_onnx(model, ONNX_PATH, DEVICE)


[Step 6] Exporting model to ONNX...


/tmp/ipykernel_2174/532232814.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(str(BEST_MODEL_PATH), map_location=DEVICE))


[ONNX] Exported to: /content/drive/MyDrive/AegisSafeRoad/training2/aegis_crash_detector.onnx  (12.4 MB)


In [ ]:
import torchvision.transforms as T

In [ ]:

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BEST_MODEL_PATH = Path(
    "/content/drive/MyDrive/AegisSafeRoad/training2/best.pt"
)

model = CrashDetector().to(DEVICE)

model.load_state_dict(
    torch.load(
        BEST_MODEL_PATH,
        map_location=DEVICE,
        weights_only=True
    )
)

model.to(DEVICE)
model.eval()

print("[OK] Model loaded")

[OK] Model loaded


In [ ]:
import cv2
import numpy as np


N_FRAMES = 16
IMG_SIZE = 224


def sample_frames_uniform(video_path):

    cap = cv2.VideoCapture(str(video_path))

    total_frames = int(
        cap.get(cv2.CAP_PROP_FRAME_COUNT)
    )

    if total_frames < N_FRAMES:
        raise ValueError(
            f"Video too short: {total_frames} frames"
        )

    indices = np.linspace(
        0,
        total_frames - 1,
        N_FRAMES,
        dtype=np.int32
    )

    frames = []

    for idx in indices:

        cap.set(
            cv2.CAP_PROP_POS_FRAMES,
            int(idx)
        )

        ok, frame = cap.read()

        if not ok:
            raise RuntimeError(
                f"Cannot read frame {idx}"
            )

        frame = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        frame = cv2.resize(
            frame,
            (IMG_SIZE, IMG_SIZE)
        )

        frames.append(frame)

    cap.release()

    return np.stack(frames)

In [ ]:
#frame = frame.astype(np.float32) / 255.0#

In [ ]:
import torchvision.transforms as T

IMAGENET_NORMALIZE = T.Normalize(
    mean=[0.485, 0.456, 0.406],
    std =[0.229, 0.224, 0.225]
)

def preprocess_video(video_path):
    frames = sample_frames_uniform(video_path)           # (16, 224, 224, 3)  uint8

    frames = frames.astype(np.float32) / 255.0           # [0, 1]  float32
    frames = np.transpose(frames, (0, 3, 1, 2))          # (16, 3, 224, 224)
    tensor = torch.from_numpy(frames)                    # float32 tensor

    # Apply ImageNet normalization per frame — exact same as CrashDataset
    tensor = torch.stack([
        IMAGENET_NORMALIZE(tensor[i]) for i in range(N_FRAMES)
    ])                                                   # (16, 3, 224, 224)

    return tensor                                        # no unsqueeze here

In [ ]:
@torch.no_grad()
def predict_video(video_path):
    x = preprocess_video(video_path)
    x = x.unsqueeze(0).to(DEVICE)

    prob = model(x).item()   # sigmoid already applied inside model — do NOT call it again

    pred  = int(prob >= 0.5)
    label = "CRASH" if pred == 1 else "NORMAL"

    return {"label": label, "probability": prob}

In [ ]:
!ls ./videos2test/

blond_gym_boy10.mp4  car_crash_09.mp4	     good_friends6.mp4
car_crash_06.mp4     car_crash_10.mp4	     gym_boy_1.mp4
car_crash_07.mp4     city_friends.mp4	     gym_boy_2.mp4
car_crash_08.mp4     friendly_gym_boy_1.mp4  walking_gym_bro01.mp4


In [ ]:
VIDEO_DIR = Path(
    "/content/videos2test"
)


In [ ]:

for video in sorted(
    VIDEO_DIR.glob("*.mp4")
):

    result = predict_video(video)

    print(
        f"{video.name:25s}"
        f" -> "
        f"{result['label']:7s}"
        f"  prob={result['probability']:.4f}"
    )

blond_gym_boy10.mp4       -> NORMAL   prob=0.0035
car_crash_06.mp4          -> CRASH    prob=0.9904
car_crash_07.mp4          -> CRASH    prob=0.7178
car_crash_08.mp4          -> CRASH    prob=0.9739
car_crash_09.mp4          -> CRASH    prob=0.8591
car_crash_10.mp4          -> NORMAL   prob=0.4610
city_friends.mp4          -> NORMAL   prob=0.4659
friendly_gym_boy_1.mp4    -> NORMAL   prob=0.0112
good_friends6.mp4         -> NORMAL   prob=0.3842
gym_boy_1.mp4             -> NORMAL   prob=0.0004
gym_boy_2.mp4             -> NORMAL   prob=0.0147
walking_gym_bro01.mp4     -> NORMAL   prob=0.0001


In [ ]:
# Diagnostic — print raw logit before sigmoid
@torch.no_grad()
def diagnose_video(video_path):
    x = preprocess_video(video_path)
    x = x.unsqueeze(0).to(DEVICE)

    # Get raw output before sigmoid
    raw = model.classifier[:-1](  # everything except final sigmoid
        model.temporal_attention(
            model.feature_extractor(
                x.view(1 * N_FRAMES, 3, 224, 224)
            ).view(1, N_FRAMES, 1280)
        )
    )

    prob = torch.sigmoid(raw).item()
    print(f"{Path(video_path).name:30s} raw_logit={raw.item():.4f}  prob={prob:.4f}")

for video in sorted(VIDEO_DIR.glob("*.mp4")):
    diagnose_video(video)

blond_gym_boy10.mp4            raw_logit=-5.6576  prob=0.0035
car_crash_06.mp4               raw_logit=4.6360  prob=0.9904
car_crash_07.mp4               raw_logit=0.9336  prob=0.7178
car_crash_08.mp4               raw_logit=3.6194  prob=0.9739
car_crash_09.mp4               raw_logit=1.8077  prob=0.8591
car_crash_10.mp4               raw_logit=-0.1563  prob=0.4610
city_friends.mp4               raw_logit=-0.1365  prob=0.4659
friendly_gym_boy_1.mp4         raw_logit=-4.4808  prob=0.0112
good_friends6.mp4              raw_logit=-0.4718  prob=0.3842
gym_boy_1.mp4                  raw_logit=-7.8348  prob=0.0004
gym_boy_2.mp4                  raw_logit=-4.2039  prob=0.0147
walking_gym_bro01.mp4          raw_logit=-9.4294  prob=0.0001


In [ ]:
tensor = np.load(
    "/content/video_tensors/val/crash_0001.npy"
)

In [ ]:
tensor = np.load(
    "/content/video_tensors/val/normal_0001.npy"
)